# Demonstrativo: Algoritmo de Huffman na Transmissão de Telemetria SCADA-Core
### Estação de Reabastecimento de Hidrogênio (HRS - Hydrogen Refueling Station)

Este notebook demonstra a implementação e simulação do **Algoritmo de Huffman** para compressão sem perdas (*lossless*) de dados de telemetria industrial em um sistema **SCADA** aplicado à **Estação de Reabastecimento de Hidrogênio (HRS)** (Setor 100 - Armazenamento em Cascata e Setor 200 - Condicionamento Térmico Chiller).

---

## 1. Contexto Operacional e Mapeamento de Tags ISA-5.1

No ambiente da estação de hidrogênio, o vetor de estado da instrumentação de segurança dos Setores 100/200 é composto por 5 chaves/sensores discretizados segundo as normas SAE J2601 e IEC 61511:

1. `PT-101`: Transmissor de Pressão do Tanque LP TK-101 (1 = Sobrepressão crítica $\ge 400\text{ bar}$)
2. `TT-101`: Transmissor de Temperatura de Parede de TK-101 (1 = Sobretemperatura $\ge 85^\circ\text{C}$)
3. `AT-101`: Detector de Gás Hidrogênio Inflamável (1 = Limite Crítico de Fuga $\ge 40\%\text{ LFL}$)
4. `PT-201`: Transmissor de Pressão da Linha do Chiller (1 = Transiente de Alta Pressão $\ge 850\text{ bar}$)
5. `ESD-100`: Botão de Parada de Emergência Geral da Estação (1 = Emergência Acionada / Trip SIS)

Sem compressão, cada amostra de telemetria exige uma palavra binária de tamanho fixo de **5 bits**.

In [ ]:
import heapq
import math
import random
import matplotlib.pyplot as plt
import pandas as pd

# Configuração para exibição limpa dos gráficos
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')

## 2. Modelo Probabilístico das Mensagens da Planta

Devido ao regime contínuo e estável da estação de hidrogênio, o estado normal seguro (`00000`) ocorre na imensa maioria das leituras ($90\%$), enquanto situações de transiente de pressão no chiller, alarmes de sobrepressão/temperatura e detecção de gás ocorrem com menor frequência operacional.

In [ ]:
# Tabela de probabilidades de cada estado lógico [PT-101, TT-101, AT-101, PT-201, ESD-100]
state_probabilities = {
    '00000': 0.900,  # Operação Nominal Estável (Condições Seguras)
    '00010': 0.050,  # Transiente de Pressão no Chiller (PT-201)
    '10000': 0.025,  # Alarme de Sobrepressão em TK-101 (PT-101)
    '01000': 0.015,  # Alarme de Sobretemperatura (TT-101)
    '00100': 0.009,  # Detector de Fuga de Hidrogênio (AT-101)
    '00001': 0.001   # Parada Geral de Emergência (ESD-100)
}

df_probs = pd.DataFrame(list(state_probabilities.items()), columns=['Vetor de Estado', 'Probabilidade'])
df_probs['Porcentagem'] = df_probs['Probabilidade'] * 100
print('--- Tabela de Probabilidades de Telemetria (Estação H2) ---')
display(df_probs)

## 3. Construção da Árvore de Huffman e Geração de Códigos

Implementamos a estrutura de árvore binária e utilizamos uma fila de prioridade (*min-heap*) para criar os códigos de prefixo de tamanho variável.

In [ ]:
class HuffmanNode:
    def __init__(self, symbol=None, freq=0, left=None, right=None):
        self.symbol = symbol
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq

def build_huffman_tree(probabilities):
    heap = [HuffmanNode(symbol=sym, freq=p) for sym, p in probabilities.items()]
    heapq.heapify(heap)

    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = HuffmanNode(freq=left.freq + right.freq, left=left, right=right)
        heapq.heappush(heap, merged)

    return heap[0]

def generate_codes(root, current_code="", code_dict=None):
    if code_dict is None:
        code_dict = {}
    if root is not None:
        if root.symbol is not None:
            code_dict[root.symbol] = current_code
        generate_codes(root.left, current_code + "0", code_dict)
        generate_codes(root.right, current_code + "1", code_dict)
    return code_dict

# Construir árvore e gerar dicionário
tree_root = build_huffman_tree(state_probabilities)
huffman_codes = generate_codes(tree_root)

print("\n--- Códigos de Huffman Gerados ---")
for sym, code in huffman_codes.items():
    print(f"Estado: {sym} | Probabilidade: {state_probabilities[sym]:6.3f} | Código Huffman: {code:>5} ({len(code)} bits)")

## 4. Simulação de Transmissão SCADA em Tempo Real

Simulamos uma transmissão contínua de **10.000 amostras** de telemetria enviadas pelo CLP de campo para o Servidor SCADA central.

In [ ]:
# Gerar série temporal de dados de amostragem do SCADA (10.000 amostras)
NUM_SAMPLES = 10000
symbols = list(state_probabilities.keys())
weights = list(state_probabilities.values())

random.seed(42)
telemetry_stream = random.choices(symbols, weights=weights, k=NUM_SAMPLES)

# 1. Codificação da transmissão em tamanho fixo (5 bits por amostra)
raw_bitstream = "".join(telemetry_stream)
raw_size_bits = len(raw_bitstream)

# 2. Codificação comprimida via Huffman
compressed_bitstream = "".join([huffman_codes[sym] for sym in telemetry_stream])
compressed_size_bits = len(compressed_bitstream)

print(f"Amostras Processadas: {NUM_SAMPLES:,}")
print(f"Tamanho Bruto (Sem Compressão): {raw_size_bits:,} bits ({raw_size_bits/8/1024:.2f} KB)")
print(f"Tamanho Comprimido (Huffman):   {compressed_size_bits:,} bits ({compressed_size_bits/8/1024:.2f} KB)")
print(f"Redução total de volume:        {100 * (1 - compressed_size_bits/raw_size_bits):.2f}%")

## 5. Algoritmo de Decodificação e Validação Sem Perdas (*Lossless*)

No receptor SCADA, o fluxo binário comprimido é navegado bit a bit através da Árvore de Huffman para reconstruir a sequência exata de telemetria.

In [ ]:
def decode_huffman(bitstream, root):
    decoded_symbols = []
    current = root
    for bit in bitstream:
        if bit == '0':
            current = current.left
        else:
            current = current.right

        if current.symbol is not None:
            decoded_symbols.append(current.symbol)
            current = root
    return decoded_symbols

# Executar decodificação no receptor SCADA
decoded_telemetry = decode_huffman(compressed_bitstream, tree_root)

# Validação formal de integridade de dados
is_equal = (telemetry_stream == decoded_telemetry)
print(f"Validação de Integridade (Lossless Verification): {'✅ SUCESSO! Recuperação 100% fiel.' if is_equal else '❌ ERRO na integridade!'}")

## 6. Análise Teórica e Gráficos de Desempenho

Comparamos os resultados experimentais com o limite teórico da **Entropia de Shannon** $H(S)$.

In [ ]:
# Entropia de Shannon H(S)
entropy = -sum(p * math.log2(p) for p in state_probabilities.values())

# Comprimento Médio Teórico L
avg_length = sum(state_probabilities[sym] * len(huffman_codes[sym]) for sym in state_probabilities)

print(f"Entropia de Shannon H(S):          {entropy:.4f} bits/símbolo")
print(f"Comprimento Médio da Palavra L:    {avg_length:.4f} bits/símbolo")
print(f"Comprimento do Formato Fixo:       5.0000 bits/símbolo")
print(f"Eficiência Teórica de Compressão:  {(1 - avg_length/5)*100:.2f}%")

# Gráfico comparativo de volume transmitido
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Visualização 1: Comparação de Bits
categories = ['Sem Compressão\n(Tamanho Fixo)', 'Com Compressão\n(Huffman SCADA)']
values_kb = [raw_size_bits / 8 / 1024, compressed_size_bits / 8 / 1024]
colors = ['#e74c3c', '#2ecc71']

bars = ax1.bar(categories, values_kb, color=colors, width=0.5)
ax1.set_ylabel('Volume de Dados (KBytes)')
ax1.set_title('Volume Total Transmitido (10.000 Amostras)')
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f} KB', ha='center', va='bottom', fontweight='bold')

# Visualização 2: Comprimento dos Códigos vs Probabilidade
states = list(state_probabilities.keys())
probs = [state_probabilities[s]*100 for s in states]
code_lens = [len(huffman_codes[s]) for s in states]

ax2.bar(states, code_lens, color='#3498db', alpha=0.8, label='Tamanho do Código (bits)')
ax2_twin = ax2.twinx()
ax2_twin.plot(states, probs, color='#f39c12', marker='o', linewidth=2, label='Probabilidade (%)')

ax2.set_xlabel('Vetor de Estado [PT-101, TT-101, AT-101, PT-201, ESD-100]')
ax2.set_ylabel('Comprimento do Código (bits)')
ax2_twin.set_ylabel('Frequência de Ocorrência (%)')
ax2.set_title('Relação Frequência vs Tamanho do Código Huffman')

plt.tight_layout()
plt.show()